# Introduction to Transformers

Welcome! In this lab, you'll learn about **transformers**, one of the core Hugging Face libraries that powers modern AI applications.

## About This Environment

You're running this notebook on an **AMD MI300 GPU** in the **AMD Developer Cloud**. The MI300 is a powerful GPU with 192GB of high-bandwidth memory, making it perfect for running large language models quickly and efficiently.

## What You'll Learn

1. [What are transformers?](#what-are-transformers)
2. [Tokenization](#tokenization)
3. [Embeddings](#embeddings)
4. [Attention](#attention)
5. [Causal language modeling](#causal-lm)
6. [Masked language modeling](#masked-lm)
7. [Putting it all together](#summary)

---

<a id="what-are-transformers"></a>

## What Are Transformers?

A transformer is a neural network architecture that processes sequences, such as text, using attention mechanisms to determine how different elements in the sequence relate to one another.

Transformers power many modern AI systems, including ChatGPT, Claude, Gemini, GitHub Copilot, and DALL-E.

### The Transformer Workflow

A simplified workflow for processing text with a transformer looks like this:

```
Text → Tokenizer → Token IDs → Embeddings → Attention → Context-Aware Representations → Task Output
```


---

<a id="tokenization"></a>

## Step 1: Tokenization (Text → Token IDs)

Transformers don't read words directly, they read tokens. That's why the workflow starts with a tokenizer to convert text into token IDs the model can process.

For example, if we start with a sentence like **"the cat is orange"**, the tokenizer:
1. Breaks it into **tokens** (small pieces of text)
2. Maps each token to a **token ID** (a number)

The exact tokens and IDs depend on the tokenizer used by the model. Let's try it out with the Qwen tokenizer:

In [ ]:
from transformers import AutoTokenizer

# Load a tokenizer (using a small Qwen3 model)
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Example sentence
text = "the cat is orange"

print(f"Using tokenizer from: {model_name}\n")
print("Original text:")
print(f"  '{text}'\n")

# Step 1: Break into tokens
tokens = tokenizer.tokenize(text)
print("Step 1 - Tokens (small pieces of text):")
print(f"  {tokens}\n")
print("  --> The 'Ġ' symbol represents a SPACE before the token")
print("      So 'Ġcat' means ' cat' (space + cat)\n")

# Step 2: Convert to token IDs
token_ids = tokenizer.encode(text)
print("Step 2 - Token IDs (numbers the model uses):")
print(f"  {token_ids}\n")

print("💡 The tokenizer converted 'the cat is orange' into numbers!")
print("   Now the transformer can process them.")

You might notice that tokenizers don't always split text into complete words. They use subword tokenization.

This is useful because:
- Common words like "the" often get their own token
- Rare words get broken into smaller pieces
- The model can handle new or uncommon words by combining known subword pieces

Let's see how different sentences get tokenized:

In [ ]:
# Compare tokenization of different sentences
sentences = [
    "the cat is orange",
    "the cat is tabby",
    "the cat is underweight"
]

print(f"Tokenization with {model_name}:\n")

for sentence in sentences:
    tokens = tokenizer.tokenize(sentence)
    token_ids = tokenizer.encode(sentence)
    
    print(f"Sentence: '{sentence}'")
    print(f"  Tokens: {tokens}")
    print(f"  Token IDs: {token_ids}\n")


### Key Takeaway

**Transformers work with token IDs, not raw text.** The tokenizer is the bridge between human-readable text and model-readable numbers.

---

<a id="embeddings"></a>

## Step 2: Embeddings (Token IDs → Vectors)

Next, the transformer takes those token IDs and converts each one into an embedding, which is a vector of numbers.

You can think of embeddings as coordinates in a multi-dimensional space. During training, the model learns to place tokens so that those used in similar contexts end up close together.

For example, tokens related to similar ideas may end up near each other in the embedding space:
- Days of the week like "Monday", "Tuesday", and "Wednesday"
- Countries like "France", "Germany", and "Spain"
- Colors like "red", "blue", and "green"

Because these tokens often appear in similar contexts, their embeddings become similar.

Let's load a model and look at its embeddings:

In [ ]:
from transformers import AutoModel
import torch

# Load a modern transformer model (Qwen3 from 2024)
model_name = "Qwen/Qwen3-0.6B"
print(f"Loading {model_name}...")
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✓ Model loaded!\n")

# Get the token IDs for our sentence
text = "the cat is orange"
token_ids = tokenizer.encode(text, return_tensors="pt")

print(f"Text: '{text}'\n")
print(f"Token IDs shape: {token_ids.shape}")
print(f"  → {token_ids.shape[1]} tokens\n")

# Get embeddings from the model's embedding layer
with torch.no_grad():
    # The embedding layer converts token IDs to vectors
    embedding_layer = model.get_input_embeddings()
    embeddings = embedding_layer(token_ids)

print(f"Embeddings shape: {embeddings.shape}")
print(f"  → {embeddings.shape[1]} tokens")
print(f"  → {embeddings.shape[2]} dimensions per token")
print(f"\n💡 Each token ID became a {embeddings.shape[2]}-dimensional vector!")

### Visualizing Embedding Similarity

Since similar words tend to have similar embeddings, we can measure how close their vectors are in the embedding space. Let's compare some words:

In [ ]:
import torch.nn.functional as F

# Get the embedding layer
embedding_layer = model.get_input_embeddings()

# Function to get embedding for a single word
def get_word_embedding(word):
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    embedding = embedding_layer(torch.tensor([[token_id]]))
    return embedding[0, 0]

# Compare words from different categories
words_animals = ["cat", "dog"]
words_colors = ["red", "blue"]
words_countries = ["India", "Canada"]

all_words = words_animals + words_colors + words_countries
word_embeddings = {word: get_word_embedding(word) for word in all_words}

print("Let's compare words from DIFFERENT categories:\n")

# Compare cat (animal) to other words
cat_embedding = word_embeddings["cat"]

print("How similar are these words to 'cat' (an animal)?\n")
for word in all_words:
    if word == "cat":
        continue
    similarity = F.cosine_similarity(
        cat_embedding.unsqueeze(0),
        word_embeddings[word].unsqueeze(0)
    ).item()
    
    category = ""
    if word in words_animals:
        category = "(animal)"
    elif word in words_colors:
        category = "(color)"
    elif word in words_countries:
        category = "(country)"
    
    print(f"  cat ↔ {word:10s} {category:10s}: {similarity:.3f}")

print("\n💡:")
print("   • 'dog' (another animal) is the most similar to 'cat'")
print("   • Colors and countries are less similar to 'cat'")
print("   • Embeddings encode aspects of a token's meaning as a vector of numbers")

<a id="attention"></a>

## Step 3: Attention

Once each token has been converted into an embedding vector, we have a row of vectors. But at this point, each embedding only represents that token in isolation. The transformer needs a way to update these embeddings so each token understands its context (i.e. the words around it).

### What is Attention?

Imagine reading the sentence: *"The chicken didn't cross the road because it was too tired."*

What does *"it"* refer to? You know it's *"the chicken"* because you connect it with earlier words in the sentence. Attention works in a similar way. It lets each word look at the other words in the sequence and decide which ones are most relevant for understanding it. By giving different levels of importance (or attention) to different words, the model can understand context and how a word’s meaning can change depending on the surrounding words.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn.functional as F

# Load model
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Compare the same word "bank" in different contexts
sentences = [
    "I went to the bank",        # bank = financial institution
    "I sat by the river bank"    # bank = edge of river
]

representations = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")

    # Get contextual representation AFTER attention
    with torch.no_grad():
        outputs = model(**inputs)

    # Find "bank" token and get its vector
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    for i, token in enumerate(tokens):
        if "bank" in token.lower():
            bank_idx = i
            break
    representations.append(outputs.last_hidden_state[0, bank_idx])

    print(f"'{sentence}'")

# Compare the two "bank" vectors
similarity = F.cosine_similarity(
    representations[0].unsqueeze(0),
    representations[1].unsqueeze(0)
).item()

print(f"\nSimilarity between the two 'bank' vectors: {similarity:.3f}")
print("\n💡 Same word, different contexts → different vectors!")
print("   Attention lets each word look at other tokens in the sequence to understand its meaning.")

### Key Takeaway

After the attention mechanism is applied, words are no longer treated in isolation, they become **context-aware**.

The same word can have different meanings depending on context. The transformer uses attention to look at other words in the sequence, then updates each word’s vector to capture its meaning in that specific sentence.

Depending on the task, these contextual representations can then be used in different ways—for example, to predict the next token, predict masked tokens, or predict a label.

Let's look at two example tasks.

---

<a id="causal-lm"></a>

## Causal Language Modeling

In causal language modeling, we take the transformer's context-aware vectors and use them to **predict what token should come next**. For example, if we give the model a story starter like "Once upon a time", it predicts a probability distribution over possible next tokens and chooses one to continue the story.

It's called causal because the model can't look ahead. At each position, it predicts the next token using only the tokens that came before it. This left-to-right constraint, only seeing past tokens and never future ones, is what makes causal language models well suited for **text generation**.

In [ ]:
# Demonstrating one-token-at-a-time generation
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Start with our prompt
text = "The weather today is"
print(f"Starting text: '{text}'\n")

# Generate 5 tokens, showing the process
for step in range(5):
    # Tokenize current text
    inputs = tokenizer(text, return_tensors="pt")
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predictions for the NEXT token
    next_token_logits = outputs.logits[0, -1, :]
    
    # Get the most likely next token
    next_token_id = next_token_logits.argmax()
    next_token = tokenizer.decode(next_token_id)
    
    # Add it to our text
    text += next_token
    
    print(f"Step {step + 1}: Added '{next_token}' → '{text}'")

print("\n💡 This is how modern language models generate text: one token at a time.")
print("   At each step, the model predicts the next token based on all previous tokens.")

### Use Cases for Causal Language Models

Causal language models (like GPT-4, Claude, and Qwen) are well suited for tasks such as:
- **Text generation**: writing stories, code, or emails
- **Chatbots**: responding in conversations
- **Code completion**: predicting the next line of code
- **Creative writing**: generating ideas and content

The key property is the **left-to-right constraint**. The model predicts each new token using only the tokens that came before it.

---

<a id="masked-lm"></a>

## Masked Language Modeling

Another common language-modeling objective is masked language modeling (MLM).

Instead of predicting the next word, MLM hides a word in a sentence and trains the model to guess the missing word. The model can use **both** the left and the right side of the sentence to figure out the masked word.

Models trained this way are especially good at language understanding tasks where you want the model to interpret text using full context, such as text classification, labeling words in a sentence (like names, dates, places), and extracting information and answering questions based on a passage.

In [ ]:
from transformers import pipeline

# Load a fill-mask model (masked language model)
# Note: We use BERT here because Qwen is a causal model and can't do masked prediction
print("Loading masked language model (BERT)...")
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
print("✓ Model loaded!\n")

# Use sentences with strong contextual clues
sentences = [
    "The sky is [MASK].",
    "I drink [MASK] in the morning."
]

for sentence in sentences:
    print(f"Sentence: '{sentence}'")
    print("Top predictions:\n")
    
    results = fill_mask(sentence, top_k=3)
    
    for i, result in enumerate(results, 1):
        word = result['token_str'].strip()
        score = result['score']
        filled = result['sequence']
        print(f"  {i}. '{word}' ({score:.1%})")
        print(f"      → {filled}")
    
    print("\n" + "="*60 + "\n")

print("💡 Masked Language Modeling uses BOTH left AND right context, it sees the FULL sentence.")
print("   This makes it great for understanding.")

<a id="summary"></a>

## Putting It All Together

Let's review the complete transformer workflow:

### The Full Pipeline

```
Input Text: "the cat is orange"
     ↓
1. TOKENIZATION
   ['the', 'cat', 'is', 'orange']
     ↓
2. TOKEN IDs
   [262, 3797, 318, 10912]
     ↓
3. EMBEDDINGS
   [vector1, vector2, vector3, vector4]
   (each is 768 dimensions)
     ↓
4. ATTENTION (multiple layers)
   - 'cat' attends to 'the'
   - 'orange' attends to 'cat' and 'is'
   - Each token updates based on context
     ↓
5. CONTEXT-AWARE REPRESENTATIONS
   [contextual_vector1, contextual_vector2, ...]
     ↓
6. TASK-SPECIFIC OUTPUT
   - Causal LM: Predict next token
   - Masked LM: Predict missing token
   - Classification: Predict label
```

And trace through the pipeline with code:

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

print("=" * 60)
print("COMPLETE TRANSFORMER PIPELINE WALKTHROUGH")
print("=" * 60)

# Setup - using Qwen3 for this demo
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "the cat is orange"

print(f"\nUsing model: {model_name}")
print(f"Input Text: '{text}'\n")

# Step 1: Tokenization
print("Step 1: TOKENIZATION")
tokens = tokenizer.tokenize(text)
print(f"  Tokens: {tokens}\n")

# Step 2: Token IDs
print("Step 2: TOKEN IDs")
token_ids = tokenizer.encode(text, return_tensors="pt")
print(f"  Token IDs: {token_ids.tolist()[0]}\n")

# Step 3: Initial Embeddings
print("Step 3: EMBEDDINGS")
with torch.no_grad():
    embedding_layer = model.get_input_embeddings()
    embeddings = embedding_layer(token_ids)
print(f"  Shape: {embeddings.shape}")
print(f"  → {embeddings.shape[1]} tokens, {embeddings.shape[2]} dimensions each\n")

# Step 4: Attention + Context
print("Step 4: ATTENTION (running through all transformer layers)")
with torch.no_grad():
    outputs = model(token_ids)
print(f"  The model has {model.config.num_hidden_layers} transformer layers")
print(f"  Each layer has {model.config.num_attention_heads} attention heads\n")

# Step 5: Context-Aware Representations
print("Step 5: CONTEXT-AWARE REPRESENTATIONS")
contextual_embeddings = outputs.last_hidden_state
print(f"  Shape: {contextual_embeddings.shape}")
print(f"  → Same shape as input embeddings, but now context-aware!\n")

# Step 6: Could use for various tasks
print("Step 6: TASK-SPECIFIC OUTPUT")
print("  These contextual representations can now be used for:")
print("  ✓ Causal LM: Predict next token (Qwen3, GPT, Claude)")
print("  ✓ Masked LM: Predict missing token (requires encoder model like BERT)")
print("  ✓ Classification: Predict sentiment, topic, etc.")
print("  ✓ Question Answering: Find answer from a passage")
print("  ✓ And many more tasks!\n")

print("=" * 60)
print("That's the complete transformer workflow!")
print("=" * 60)

## Summary: What You've Learned

Congratulations! You've now run transformers on an AMD MI300 GPU. 🎉 

### The Transformer Workflow

1. **Tokenization**: Text → Token IDs
2. **Embeddings**: Token IDs → Vectors
3. **Attention**: Vectors look at each other and become context-aware
4. **Multiple Layers**: Repeated attention passes refine understanding
5. **Task Output**: Use representations for different tasks

### Two Key Tasks

**Causal Language Modeling**:
- Predict the NEXT token
- Can only look left (before current position)
- Used for: text generation, chatbots, code completion
- Examples: GPT-4, Claude, Qwen3
- **We used Qwen3-0.6B in this notebook!**

**Masked Language Modeling**:
- Predict MISSING tokens in the middle
- Can look both left AND right (full context)
- Used for: classification, understanding, question answering
- Examples: BERT, DeBERTa, RoBERTa
- **Requires different architecture than causal models**


### Keep Experimenting!

Try these challenges:
- Load a different model from Hugging Face and analyze its attention patterns
- Compare how causal and masked models handle the same sentence
- Visualize embedding spaces for different word categories

**You're now ready to work with transformer models!** 🚀